# Cas d'une bulle isolée dans un domaine tripériodique


## Introduction
 
 Validation made by: Grégoire Husser

 Report generated 03/06/2025

In [ ]:
from trustutils import run 
from pathlib import Path
 
run.TRUST_parameters("1.9.6")

In [ ]:
run.reset()
run.initBuildDirectory()  # copy src/ in build/
run.addCase(".", "bulle_isolee_ijk_counter.data", nbProcs=8) 
run.addCase(".", "bulle_isolee_disc_counter.data", nbProcs=8) 
run.printCases()
run.runCases()

## Computer Performance

In [ ]:
run.tablePerf()

## Resultats


In [ ]:
from trustutils import visit
import os
print(os.getcwd())
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
def param_image(name, grid=True, legend=False, legend_out=True, n_dpi=100, fontsizelegend=10):
    scalepng=3.5       
    inSizeLegend=int(scalepng*3.5)
    if (grid):
        plt.grid()
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.tick_params(axis='x', labelsize=12)
    plt.tick_params(axis='y', labelsize=12)
    if (legend_out):
    	plt.legend(fontsize=fontsizelegend, loc='center left', bbox_to_anchor=(1.05, 0.5))
    else:
    	plt.legend(fontsize=fontsizelegend, loc=0,prop={'size':inSizeLegend}, frameon=True, ncol=1)
    	plt.legend(fontsize=fontsizelegend).set_visible(legend) ### desactive la legende

On réalise un cas test de déplacement d'une bulle isolée dans un domaine tripériodique. On se base pour cela sur les travaux de A. Esmaeeli et G. Tryggvason [[1]](https://www.researchgate.net/publication/245233905_Direct_numerical_simulations_of_bubbly_flows_Part_1_Low_Reynolds_number_arrays). 

On mesure le Reynolds de bulle défini tel que :
\begin{equation}
Re_b = \frac{\rho_f d_e W_b}{\mu_f}
\end{equation}

Afin de pouvoir comparer nos résultats à [[1]](https://www.researchgate.net/publication/245233905_Direct_numerical_simulations_of_bubbly_flows_Part_1_Low_Reynolds_number_arrays) pour un taux de vide $\alpha = 0.0335$, on doit avoir :
\begin{equation}
\gamma = \frac{\rho_b}{\rho_f}=0.05,  ~\lambda = \frac{\mu_b}{\mu_f}=0.05, ~N=\frac{\rho_f^2d_e^3g}{\mu_f^2}=1000^{1/2} ~et~ Eo=\frac{\rho_fgd_e^2}{\sigma}=1.0
\end{equation}

On prend pour cela les valeurs physiques suivantes :
- $\rho_f = 880~kg/m^3$
- $\mu_f = 0.041~Pa.s$
- $\rho_g = 44~kg/m^3$
- $\mu_g = 0.0020~kg/m^3$
- $\sigma = 0.031~N/m$
- $d_b = 1.9~mm$

On ne dispose pas des données nécessaires pour étudier le transitoire de la bulle. L'article nous permet juste de conclure sur la vitesse terminale de bulle, avec $Re_{b, Esmaeeli}=1.35$.

Dans l'optique de comparer les résultats entre les TrioIJK et Trio Discontinu, on ajoute une vitesse à contre-courant de l'écoulement. En empêchant la bulle de traverser le domaine, on peut effectuer un calcul tripériodique sur Trio Discontinu. On choisit cette vitesse à $-0.0335~m/s$.


In [ ]:
rhof = 880
muf = 0.041
rhog = 44
mug = 0.002
sigma = 0.031
d = 1.9 * 10**-3
v_counter = -0.0335

### Calcul FT IJK

In [ ]:
def read_centre(file_path):                                                      
    df = pd.read_csv(
        file_path,
        sep=r'\s+',
        header=None,
        usecols=[0, 1],
        names=['time', 'pos'],
        skiprows=0
    )                                                                       
    return df  

In [ ]:
ijk_centre = os.path.join('build', 'bulle_isolee_ijk_counter_bulles_centre_x.out')
df_ijk_centre = read_centre(ijk_centre)

df_ijk_centre['vel_bubble'] = np.gradient(df_ijk_centre['pos'], df_ijk_centre['time']) - v_counter
df_ijk_centre['Re'] = df_ijk_centre['vel_bubble'] * rhof * d / muf

### Calcul VDF FT Discontinu

In [ ]:
def read_compo(file_path):                                                      
    df_time = pd.read_csv(
        file_path,
        sep=r'\s+',
        header=None,
        usecols=[1],
        names=['time'],
        skiprows=lambda x: x%2 == 1
    )
    df_vel = pd.read_csv(
        file_path,
        sep=r'\s+',
        header=None,
        usecols=[7],
        names=['vel'],
        skiprows=lambda x: x%2 == 0
    )
    return df_time.join(df_vel)

In [ ]:
disc_compo = os.path.join('build', 'composantes_connexes.txt')
df_disc_compo = read_compo(disc_compo)

df_disc_compo['vel_bubble'] = df_disc_compo['vel'] - v_counter
df_disc_compo['Re'] = df_disc_compo['vel_bubble'] * rhof * d / muf

### Analyse des résultats

In [ ]:
plt.figure()
plt.plot(df_ijk_centre['time'], df_ijk_centre['Re'], label=r'$Re_{b,ijk}$')
plt.plot(df_disc_compo['time'], df_disc_compo['Re'], label=r'$Re_{b,discontinu}$')
plt.hlines(1.35, 0, max(df_ijk_centre['time']), label=r'$Re_{b,Esmaeeli}$', color='red', linestyles='--')
plt.xlabel('t')
plt.ylabel(r'$Re_b$')
plt.ylim(0,1.5) # Limite affichage des oscillations
param_image('Re.png')
plt.show()